In [ ]:
import requests
import os
from datetime import datetime

# Base URL for ADA score PDFs
base_url = "https://adaction.org/wp-content/uploads/"

# Create a directory to store the downloaded PDFs
os.makedirs("downloaded_pdfs", exist_ok=True)

# Define year-specific URLs based on the actual pattern
# Format: {year: (upload_year, upload_month)}
year_urls = {
    2010: (2017, 1),
    2011: (2017, 1),
    2012: (2017, 1),
    2013: (2017, 1),
    2014: (2017, 1),
    2015: (2017, 1),
    2016: (2017, 1),
    2017: (2018, 1),
    2018: (2019, 2),
    2019: (2020, 2),
    2020: (2021, 3),
    2021: (2022, 2),
    2022: (2022, 12),
}

# Try to download PDFs for each year
successful_downloads = 0
failed_downloads = 0

for year, (upload_year, upload_month) in year_urls.items():
    pdf_filename = f"{year}.pdf"
    pdf_path = os.path.join("downloaded_pdfs", pdf_filename)
    
    # Skip if file already exists
    if os.path.exists(pdf_path):
        file_size = os.path.getsize(pdf_path)
        if file_size > 1000:  # Only skip if file is reasonably sized (> 1KB)
            print(f"✓ {pdf_filename} already exists ({file_size:,} bytes), skipping...")
            successful_downloads += 1
            continue
    
    # Construct URL
    pdf_url = f"{base_url}{upload_year}/{upload_month:02}/{year}.pdf"
    
    try:
        print(f"Downloading {pdf_filename} from {pdf_url}...", end=" ")
        response = requests.get(pdf_url, timeout=30)
        
        # Check if the request was successful
        if response.status_code == 200:
            with open(pdf_path, "wb") as pdf_file:
                pdf_file.write(response.content)
            file_size = len(response.content)
            print(f"✓ Success! ({file_size:,} bytes)")
            successful_downloads += 1
        else:
            print(f"✗ Failed! Status code: {response.status_code}")
            failed_downloads += 1
            
    except requests.exceptions.RequestException as e:
        print(f"✗ Error: {e}")
        failed_downloads += 1

print(f"\n{'='*60}")
print(f"Download Summary:")
print(f"  Successful: {successful_downloads}")
print(f"  Failed: {failed_downloads}")
print(f"  Total: {successful_downloads + failed_downloads}")
print(f"{'='*60}")

# If there were failures, try alternative search
if failed_downloads > 0:
    print("\nNote: If some downloads failed, you may need to manually find the correct URLs.")
    print("The PDFs might be located at different year/month combinations on the website.")